In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F

application_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_VOLUME}/{FILES['application_train']}")
)


In [0]:
display(application_raw.limit(10))

In [0]:
bureau_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_VOLUME}/{FILES["bureau"]}")
)

In [0]:
display(bureau_raw.limit(10))

In [0]:
display(FILES)

#### Passo 1.2 - Criar uma função de leitura e ingerir todas as fontes

Objetivo: Evitar repetir o mesmo bloco de leitura cinco vezes.
Por que isso importa em crédito: Funções tornam o pipeline consistente e reduzem diferenças acidentais entre fontes.


In [0]:
def read_home_credit_csv(table_name: str):
    file_name = FILES[table_name]
    path = f"{RAW_VOLUME}/{file_name}"

    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
    )



In [0]:
bureau_raw = read_home_credit_csv("bureau")
previous_raw = read_home_credit_csv("previous_application")
installments_raw = read_home_credit_csv("installments_payments")
credit_card_raw = read_home_credit_csv("credit_card_balance")

#### Passo 1.3 - Salvar a camada Bronze como Delta

Primeiro armazenamos os arquivos CSV brutos dentro de um Volume. Depois, no notebook de ingestão, lemos esses arquivos como DataFrames Spark e salvamos cada fonte como uma tabela Delta na camada Bronze. Assim, passamos a consumir os dados por nome de tabela dentro do Databricks, sem precisar reler o CSV bruto toda vez

In [0]:
bronze_frames = {
    "application_train": application_raw,
    "bureau": bureau_raw,
    "previous_application": previous_raw,
    "installments_payments": installments_raw,
    "credit_card_balance": credit_card_raw
}

for table_name, df in bronze_frames.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(f"{BRONZE}.{table_name}")
    )
    print("gravada:", f"{BRONZE}.{table_name}")

In [0]:
%sql
SHOW TABLES IN credit_lab.bronze


####  Passo 1.4 - Criar o inventário de volume e schema

Objetivo: Registrar linhas, colunas e chaves antes de transformar.
Por que isso importa em crédito: Mudanças inesperadas de volume ou schema podem indicar arquivo incompleto, versão errada ou problema de ingestão.

In [0]:
tables_to_profile = [
    "application_train",
    "bureau",
    "previous_application",
    "installments_payments",
    "credit_card_balance"
]

inventory_rows = []

for table_name in tables_to_profile:
    df = spark.table(f"{BRONZE}.{table_name}")
    inventory_rows.append((
        table_name,
        df.count(),
        len(df.columns),
    ))

# A tupla é uma linha imutável (se estivesse entre chaves, seria uma lista) ⬆️

    inventory = spark.createDataFrame(
        inventory_rows,
        ["table_name", "row_count", "column_count"]
    )

    display(inventory)

In [0]:
(
    inventory.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{MONITORING}.source_inventory")
)

# source_inventory é o nome da tabela que vai gravar o resultado da query.

In [0]:
%sql
SHOW TABLES in credit_lab.monitoring

#### Passo 1.5 - Confirmar a granularidade e as chaves

Objetivo: Responder o que uma linha representa em cada fonte.
Por que isso importa em crédito: Sem granularidade, um join pode multiplicar clientes e distorcer exposição, atraso e bad rate.


Leitura direta do CSV x leitura da tabela Bronze: No início da ingestão, usamos funções como read_home_credit_csv() para ler diretamente os arquivos CSV armazenados no Volume e criar DataFrames temporários, como bureau_raw e previous_raw. Depois, esses DataFrames são persistidos como tabelas Delta na camada Bronze. A partir desse momento, os próximos passos passam a ler as tabelas Bronze com spark.table(...), porque elas se tornam a fonte oficial e persistente da pipeline. Tecnicamente, poderíamos continuar usando os DataFrames _raw na mesma sessão, mas ler novamente a partir da Bronze garante que estamos trabalhando com o que foi efetivamente salvo e permite retomar o projeto em outra sessão sem precisar reler os CSVs originais.

In [0]:
application = spark.table(f"{BRONZE}.application_train")
bureau = spark.table(f"{BRONZE}.bureau")
previous = spark.table(f"{BRONZE}.previous_application")
installments = spark.table(f"{BRONZE}.installments_payments")
credit_card = spark.table(f"{BRONZE}.credit_card_balance")

checks = [
    ("application", application.count(), application.select("SK_ID_CURR").distinct().count()),
    ("bureau", bureau.count(), bureau.select("SK_ID_BUREAU").distinct().count()),
    ("previous", previous.count(), previous.select("SK_ID_PREV").distinct().count()),
    ("installments", installments.count(), installments.select("SK_ID_PREV").distinct().count()),
    ("credit_card", credit_card.count(), credit_card.select("SK_ID_CURR").distinct().count())

]
display(spark.createDataFrame(checks, ["table", "rows", "distinct_key"]))

# Podemos entender que a chave analisada é única em cada uma dessas tabelas 💡

In [0]:
display((
    installments.groupBy("SK_ID_PREV")
    .count()
    .orderBy(F.desc("count"))
)
)
# Existem X registros na tabela installments_payments associados a operação Y.

display((
    credit_card
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("*")
            .alias("rows"),
        F.countDistinct("MONTHS_BALANCE")
            .alias("months"),
    )
    .orderBy(F.desc("rows"))

# Este cliente possui X registros na tabela de credit_card distribuídos em Y valores distintos de MONTH BALANCE. O que é provável que o cliente tenha mais de um contrato/cartão (SK_ID_PREV)

))

#### Passo 1.6 - Medir cobertura com left_semi e left_anti

**Objetivo**: Quantificar quem possui ou não histórico em cada fonte.
**Por que isso importa em crédito**: Ausência de histórico pode ser informação; não deve ser confundida com nulo criado por erro de join.

**Observação**: a tabela bureau não representa apenas restrições. Ela contém histórico externo de crédito do cliente, incluindo exposição, créditos ativos, dívida e atraso. Portanto, ausência de registro em bureau significa ausência de histórico disponível nessa fonte, e não necessariamente um sinal positivo.


In [0]:
population = application.select("SK_ID_CURR").distinct()
bureau_ids = bureau.select("SK_ID_CURR").distinct()
previous_ids = previous.select("SK_ID_CURR").distinct()

coverage = [
("bureau_com_historico", population.join(bureau_ids, "SK_ID_CURR", "left_semi").count()),
("bureau_sem_historico", population.join(bureau_ids, "SK_ID_CURR", "left_anti").count()),
("previous_com_historico", population.join(previous_ids, "SK_ID_CURR", "left_semi").count()),
("previous_sem_historico", population.join(previous_ids, "SK_ID_CURR", "left_anti").count()),

]

display(spark.createDataFrame(coverage, ["metric", "customers"]))

# o left semi retorna as linhas da tabela A que possuem correspondência com a tabela B
# o left anti retorna as linhas da tabela A que não possuem correspondência com a tabela B

In [0]:
application.count()